[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc3_ml/cours/seance3_cours.ipynb)

# Séance 3.3 — The Inbox Problem (1/2) — du mot au sens

**Cours** · durée : 2h (démonstration : l'enseignant déroule, vous suivez)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer ce qu'est un *embedding* à un collègue, sans une seule formule
- dire pourquoi une recherche par mot-clé rate « charged twice » quand on cherchait « fee »
- retrouver les messages qui parlent d'un sujet, même sans aucun mot en commun
- transformer 5 000 messages en une carte de thèmes nommés, lisible par un dirigeant
- lire cette carte avec méfiance — et dire ce qu'elle ne montre pas

## Vous venez de reprendre le service client

Vous venez d'être nommé à la tête des opérations clients d'une banque en ligne.
Le trimestre dernier, **5 000 clients vous ont écrit** — réclamations,
questions, résiliations, remerciements. Personne ne les a lus.

Vendredi, votre direction veut des réponses à trois questions. On est lundi.
Vous avez Python et un après-midi.

> ### Les trois questions
>
> **1. Qu'est-ce qu'ils disent, au juste ?**
> Pas « plutôt négatif ». Des thèmes nommés, avec des volumes, qu'un dirigeant
> peut mettre dans une diapositive. → *séance 3.3*
>
> **2. Peut-on arrêter de tout lire à la main ?**
> Quelle part de la boîte peut être triée automatiquement, et avec quel taux
> d'erreur ? → *séance 3.4*
>
> **3. Combien ça coûte, et peut-on s'y fier ?**
> Des euros par mois, des taux d'erreur réels, et une recommandation
> défendable. → *séance 3.4*

### Comment cette séance fonctionne

C'est une **démonstration**. Il n'y a pas de feuille d'exercices : votre
enseignant déroule le notebook, vous suivez sur votre copie. Deux habitudes à
prendre quand même, et elles rapportent gros :

- **Avant chaque cellule importante, écrivez votre prédiction** dans une
  cellule de texte. Vingt secondes. Se tromper *par écrit* est ce qui fait
  tenir le résultat.
- Les lignes marquées **`# ← a changer`** sont pour vous. Vous ne débuggez
  rien : vous changez une décision et vous regardez ce qui bouge.

## Module 0 — Installer l'atelier

Deux paquets à installer. Mesuré le 31/08/2026, sur une machine portant déjà
le socle Colab (numpy, pandas, scikit-learn, matplotlib, **torch**) :

| Installation | Ce qu'elle ajoute | Écrase un paquet existant ? |
|---|---|---|
| `sentence-transformers` | 23 paquets, ~23 Mo | **non** |
| `google-genai` | 24 paquets, ~10 Mo | **non** |
| le modèle `all-MiniLM-L6-v2` | **91 Mo**, au premier appel | — |

Rien ne transite par votre connexion : c'est la machine virtuelle Colab qui
télécharge, et chacun a la sienne.

In [ ]:
%pip install -q sentence-transformers "google-genai==2.9.0"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from google import genai
from google.colab import userdata

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc3_ml/data/"

### Votre clé API — à faire maintenant, pas dans deux heures

1. ouvrir **https://aistudio.google.com/**, se connecter avec son compte Google
2. créer une clé API et la copier
3. dans Colab, ouvrir le panneau **Secrets** (l'icône 🔑 à gauche)
4. ajouter un secret nommé **exactement** `GEMINI_API_KEY`, y coller la clé
5. autoriser ce notebook à y accéder

> ⚠️ **Une clé ne se colle jamais dans une cellule de code.** Les notebooks se
> partagent, et c'est comme ça que les clés fuitent. Trente secondes de
> méthode, une vraie valeur professionnelle.

In [ ]:
# Un appel minuscule, uniquement pour verifier que la cle repond.
# Le faire MAINTENANT : une cle cassee se repare pendant l'exercice suivant.
ai = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
modele_llm = "gemini-3.5-flash-lite"

essai = ai.interactions.create(model=modele_llm, input="Reponds uniquement : OK")
print("cle valide ->", essai.output_text.strip())

In [ ]:
tickets = pd.read_csv(BASE + "tickets.csv")   ## 5 000 messages reels

print(tickets.shape)
tickets.head(3)

Trois colonnes : le **message** tel que le client l'a écrit, son **intention**
d'origine (77 valeurs possibles), et l'**équipe** qui devrait le traiter — huit
équipes, obtenues en regroupant les 77 intentions.

> 💡 Ce regroupement de 77 en 8 n'est pas un détail technique : c'est
> l'organigramme de votre service client. Quelqu'un a décidé que « frais de
> retrait » et « frais de virement » vont à la même équipe. C'est une décision
> de gestion, et elle est discutable.

> 📄 Les messages sont en **anglais** : c'est un corpus réel et étiqueté
> (banking77), et il n'en existe pas d'équivalent public en français. Le
> raisonnement, lui, ne dépend pas de la langue.

### L'exercice du chronomètre

Lisez les cinq messages ci-dessous et classez-les à la main dans l'une des huit
équipes. **Chronométrez-vous.**

In [ ]:
echantillon = tickets.sample(5, random_state=7)   ## ← a changer : la graine

for i, ligne in enumerate(echantillon.itertuples(), 1):
    print(f"{i}. {ligne.message}")

print()
print("Les huit equipes :", sorted(tickets["equipe"].unique()))

In [ ]:
secondes = 20   ## ← a changer : votre temps reel par message

heures = len(tickets) * secondes / 3600
print(f"{len(tickets)} messages x {secondes} s = {heures:.1f} heures de lecture")
print(f"soit {heures / 7:.1f} journees de travail, pour UNE personne")

> ### 🎯 Au tableau : **27,8 heures**
>
> Trois semaines et demie de travail à mi-temps, pour un seul trimestre, juste
> pour *lire*. C'est l'ennemi de tout l'après-midi.

## Module 1 — Compter des mots n'est pas comprendre

La méthode évidente d'abord : compter.

In [ ]:
mots = (tickets["message"].str.lower()
        .str.replace(r"[^a-z ]", " ", regex=True)
        .str.split().explode())

mots.value_counts().head(8)

`i`, `my`, `to`, `a`, `the`… et `card`. Ça ressemble à de l'analyse. Ça ne dit
rien qu'un dirigeant puisse décider.

Essayons plus finement : cherchons un thème par son mot-clé.

In [ ]:
terme = "fee"   ## ← a changer : proposez un mot-cle

trouves = tickets["message"].str.contains(terme, case=False)
print(f"'{terme}' apparait dans {trouves.sum()} messages")
tickets[trouves]["message"].head(3).tolist()

271 messages. **Combien en avons-nous raté ?** Écrivez votre prédiction avant
d'exécuter la cellule suivante.

In [ ]:
# Les memes clients, les memes problemes, d'autres mots
autres = ["charged twice", "double payment", "extra charge", "wrong amount"]

paraphrases = tickets["message"].str.contains("|".join(autres), case=False)
manques = paraphrases & ~trouves

print(f"{paraphrases.sum()} messages parlent du meme probleme autrement")
print(f"dont {manques.sum()} que la recherche sur '{terme}' n'a PAS trouves")
tickets[manques]["message"].head(4).tolist()

**44 messages, dont 43 invisibles à la recherche.** Et cette liste de
paraphrases, c'est nous qui l'avons écrite — il aurait fallu penser à
l'avance à toutes les façons de dire la même chose.

> ### 🎯 Au tableau
> Il nous faut une machine qui rapproche le **sens**, pas l'orthographe.

## Module 2 — Le sens comme coordonnées

Imaginez une carte. Pas une carte géographique : une carte du **sens**. Chaque
message y occupe une position, et deux messages qui veulent dire la même chose
atterrissent **au même endroit** — même s'ils n'ont aucun mot en commun.

C'est exactement ce que fabrique un **embedding** : il transforme une phrase en
une position. Ici, une position à 384 coordonnées au lieu de 2 — c'est la seule
différence avec la carte du tableau.

Le modèle qui fait ça est gratuit, pèse 91 Mo, et tourne sur le processeur de
votre machine virtuelle. **Aucune clé, aucun appel réseau** : cette moitié de
la séance fonctionne même si l'API est en panne.

In [ ]:
encodeur = SentenceTransformer("all-MiniLM-L6-v2")   ## 91 Mo, une fois

# 5 000 messages -> 5 000 positions a 384 coordonnees. Environ une minute.
E = encodeur.encode(tickets["message"].tolist(), batch_size=64,
                    show_progress_bar=True)

print("forme :", E.shape)   ## (5000, 384)

### Mesurer une distance sur cette carte

La **similarité cosinus** répond à une seule question : ces deux positions
sont-elles proches ? Elle vaut 1 pour deux phrases au même endroit, 0 pour deux
phrases sans rapport.

In [ ]:
def proximite(a, b):
    """A quel point ces deux phrases veulent-elles dire la meme chose ?"""
    va, vb = encodeur.encode([a, b])
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

print(round(proximite("I was charged twice for the same purchase",
                      "There is a double payment on my card"), 3))
print(round(proximite("I was charged twice for the same purchase",
                      "When will my new card arrive?"), 3))

**0,65 contre 0,21.** Les deux premières phrases n'ont **pas un mot en commun**
— ni *charged*, ni *twice*, ni *double* — et la machine les rapproche quand
même. La troisième parle d'autre chose, et elle s'éloigne.

Aucune formule n'a été montrée. Les deux nombres suffisent.

### La boîte de recherche

Maintenant, la vraie utilité : décrire un problème **dans vos mots** et
retrouver les messages qui en parlent.

In [ ]:
def chercher(question, k=5):
    """Les k messages les plus proches du sens de `question`."""
    v = encodeur.encode([question])[0]
    sim = E @ v / (np.linalg.norm(E, axis=1) * np.linalg.norm(v))
    haut = np.argsort(-sim)[:k]
    return pd.DataFrame({"proximite": sim[haut].round(3),
                         "message": tickets["message"].iloc[haut].values,
                         "equipe": tickets["equipe"].iloc[haut].values})

chercher("I still have not received my refund")

Les cinq messages remontés parlent tous d'un remboursement qui n'arrive pas, et
la colonne `equipe` le confirme. Regardez surtout : **la formulation exacte de
la question n'apparaît dans aucun d'eux.**

À vous. Cherchez un thème dont vous soupçonnez l'existence.

In [ ]:
ma_question = "my card was swallowed by the machine"   ## ← a changer

chercher(ma_question)

In [ ]:
# Et ceci fonctionne, alors que le corpus est entierement en anglais
chercher("on m'a preleve deux fois", k=3)

> 💡 Le modèle a été entraîné sur une centaine de langues. La question française
> et les réponses anglaises atterrissent au même endroit sur la carte du sens.
> C'est le même mécanisme, appliqué à travers les langues.

## Module 3 — La carte de l'inbox

384 coordonnées, ça ne se regarde pas. On les écrase en 2 pour pouvoir tracer.

In [ ]:
projection = PCA(n_components=2, random_state=42)
XY = projection.fit_transform(E)

part = projection.explained_variance_ratio_.sum()
print(f"les 2 axes traces retiennent {100 * part:.1f} % de l'information")

> ⚠️ **15,4 %.** Lisez ce chiffre avant de regarder la carte. Écraser 384
> dimensions en 2, c'est projeter une ombre : deux messages collés à l'écran
> peuvent être très loin l'un de l'autre dans le vrai espace. La carte sert à
> **explorer**, jamais à conclure. Tous les calculs qui suivent se font sur les
> 384 dimensions, pas sur l'ombre.

In [ ]:
k = 8   ## ← a changer : essayez 3, puis 8, puis 20

# k-means, exactement comme en seance 3.2 — mais sur des coordonnees de SENS
# au lieu de coordonnees d'achat. Le calcul se fait sur les 384 dimensions.
groupes = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(E)
tickets["groupe"] = groupes

plt.figure(figsize=(7, 5))
plt.scatter(XY[:, 0], XY[:, 1], c=groupes, s=6, alpha=0.5, cmap="tab10")
plt.title(f"Les 5 000 messages, projetes en 2D et regroupes en {k} themes")
plt.xlabel("axe 1")
plt.ylabel("axe 2")
plt.tight_layout()
plt.show()

tickets["groupe"].value_counts().sort_index()

Des blobs. On ne sait pas encore de quoi ils parlent — et les lire un par un
nous ramènerait aux 27,8 heures.

### Le LLM comme assistant d'étiquetage

C'est ici qu'un modèle de langue devient utile, dans son emploi le plus
modeste : pour chaque groupe, on lui donne 20 messages et on lui demande un
nom. **Huit appels d'API en tout.**

In [ ]:
def nommer(groupe, n=20):
    """Fait nommer un theme par le LLM, a partir de n messages du groupe."""
    extraits = tickets.query("groupe == @groupe")["message"].head(n).tolist()
    consigne = (
        "Voici des messages recus par le service client d'une banque en ligne.\n"
        "Donne un nom de theme en QUATRE mots maximum, puis une phrase "
        "decrivant ce que ces clients veulent.\n"
        "Reponds en francais, au format : NOM | phrase\n\n"
        + "\n".join("- " + m for m in extraits))
    return ai.interactions.create(model=modele_llm, input=consigne).output_text.strip()

for g in sorted(tickets["groupe"].unique()):
    print(f"[{g}] ({(tickets['groupe'] == g).sum():>4} messages) {nommer(g)}")

Les blobs ont des noms. C'est la première chose de cette séance qu'un
dirigeant peut lire.

> ⚠️ **Le LLM a nommé, il n'a pas décidé.** Il n'a vu que 20 messages sur
> plusieurs centaines, et il ne dit jamais « je ne suis pas sûr ». Relisez
> chaque nom en regardant les messages du groupe : c'est vous qui validez.

In [ ]:
# Le graphique que la direction lira : des themes nommes, avec des volumes.
# Remplacez les etiquettes par les noms produits ci-dessus.
noms = {g: f"theme {g}" for g in sorted(tickets["groupe"].unique())}   ## ← a changer

volumes = tickets["groupe"].value_counts().sort_values()
volumes.index = [noms[g] for g in volumes.index]

volumes.plot(kind="barh", figsize=(7, 4))
plt.title("Volume par theme, trimestre ecoule")
plt.xlabel("nombre de messages")
plt.ylabel("")
plt.tight_layout()
plt.show()

> ### 🎯 Au tableau : **question 1 répondue**
>
> Huit thèmes nommés, avec leurs volumes. Pas « plutôt négatif » : des noms et
> des nombres.

### Avant de partir — l'échec à regarder en face

Relancez la cellule de `KMeans` avec `random_state=7` au lieu de 42. Les
frontières bougent. Les messages qui changent de groupe sont ceux qui se
trouvent **entre** deux thèmes.

> La machine n'est pas perdue : **c'est vous.** Ces messages-là sont ceux pour
> lesquels votre entreprise n'a pas de catégorie. C'est une information de
> gestion, pas un défaut de l'algorithme.

---

La séance 3.4 s'attaque aux deux questions qui restent : peut-on arrêter de
tout lire à la main, et combien ça coûte.

---

## Le tableau de traduction

C'est la page à garder. Ces phrases sont celles à répéter à un collègue —
c'est le vrai test de la séance.

| Le mot | Ce qu'il veut dire |
|---|---|
| **embedding** | une carte où le sens est une position. Deux phrases qui veulent dire la même chose atterrissent dans le même quartier, même sans aucun mot en commun. |
| **similarité cosinus** | à quelle distance deux choses se trouvent sur cette carte. C'est tout. |
| **clustering (k-means)** | tracer des cercles autour des quartiers. Vous choisissez combien de cercles ; la machine choisit où. |
| **prompt** | une fiche de poste pour un intérimaire brillant qui oublie tout entre deux tâches et n'admet jamais qu'il est perdu. |

## Ce qui est sur le tableau à la fin de la séance

- **27,8 heures** de lecture manuelle. C'est l'ennemi.
- **Compter les mots ne suffit pas** : `fee` trouve 271 messages et rate 43
  des 44 messages qui parlent du même problème avec d'autres mots.
- **La question 1 est répondue** : huit thèmes nommés, avec leurs volumes.

La séance 3.4 répond aux deux autres : peut-on arrêter de tout lire à la main,
et combien ça coûte.